# Esercizio 1
Grafo orientato e non pesato.
Definire il metodo `raggiungibile(G,x1,x2,x3)` che restituisce `True` se esiste un cammino da `x1` a `x3` ed esiste un cammino da `x2` a `x3`

In [ ]:
from EsempiAlgoritmiPython.grafi.algosugrafi import visitaampiezza
from EsempiAlgoritmiPython.grafi.algosugrafipesati import Dijkstra
from backtracking import backtracking
from grafi.algosugrafipesati import floyd


def raggiunto(G,x):
    visitaampiezza(G,x)

def raggiungibile(G,x1,x2,x3):
    if x3 not in raggiunto(G,x1): return False
    if x2 not in raggiunto(G,x1): return False
    return True

## Complessità
La complessità è dominata dalla visita
### Lista di adiacenza
**Caso migliore**
Dobbiamo comunque un ciclo for
$$
CTM(n,m) = \Theta(n)
$$
$$
CSM(n,m)=\Theta(n)
$$

**Caso peggiore**
Abbiamo il costo della visita
$$
CTP(n,m) = \Theta(n+m)
$$
$$
CSP(n,m)=\Theta(n)
$$
### Matrice di adiacenza
cambia solo il costo della visita e di conseguenza la complessità

---
# Esercizio 2
G non orientato e non pesato ma connesso.

`diametro(G)` deve restituire la massima distanza tra due nodi raggiungibili, la lunghezza del più lungo cammino minimo.
Non ha senso usare Dijkstra dato che non è pesato.
Possiamo usare una visita rimodulata.

In [ ]:
def distanze(G,s): # dato s avremo il vettore delle sue distanze
    dist = [-1 for _ in range(G.n)]
    dist[s] = 0
    coda = []
    coda.append(s)
    while coda:
        u = coda.pop(0)
        for v in G.adiacenza(u):
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                coda.append(v)
    return dist

def diametro(G):
    diam = 0
    for i in range(G.n):
        dist = distanze(G,i)
        for j in dist:
            if dist[j] > diam:
                diam = dist[j]
    return diam

## Complessità
Per ogni for richiamiamo distanze, distanze inizializza il vettore

È connesso, non ci sono casi in cui i nodi sono isolati, quindi caso migliore e caso peggiore sono uguali.
$$
CTM(n,m)=CTP(m,n) = \Theta(n(n+m))
$$
$$
CSM(n,m) = CSP(n,m) = \Theta(n)
$$

---
# Esercizio 3
`G` è un grafo non orientato con pesi non negativi e `k` è una lista di nodi "ancora"

Definire `distanza_ancora(G,k)` che calcola la distanza minima tra tutti i nodi approssimandoli con la lista ancora
$$
\text{appr}(x,y) = \frac{1}{|k|} \sum_{k \in K} \text{dist}(x,k) + dist(y,k)
$$
dove la distanza è il cammino minimo tra $x$ e $k$


In [ ]:
def distanza_sorgente(G,s):
    dist = [float("inf") for _ in range(G.n)]
    dist[s] = 0
    albero = Dijkstra(G,s)
    for padre, nodo, distanza in albero:
        dist[nodo] = distanza
    return dist

def distanza_ancora(G,K):
    app = {} # inizializzo un dizionario di approssimazione
    for x in range(G.n):
        for y in range(x+1,G.n):
            app[(x,y)] = 0
    for k in K:
        dist = distanza_sorgente(G,k)
        for x in range(G.n):
            for y in range(x+1,G.n):
                app[(x,y)] += dist[x] + dist[y] # striamo calcolando la sommatoria
    for (x,y) in app:
        app[(x,y)] = app[(x,y)]/len(K)
    return app

## Complessità
È dominato dalla complessità di Dijkstra, se $r = |K|$
$$CTM(n,m,r) = CTP(n,m,r) = \Theta(r \cdot (n+m\log m))$$
Per la spaziale devo creare il dizionario, quindi in entrambi i casi
$$CSP(n,m,r) = CSM(n,m,r) = \Theta(n^2)$$

---
# Esercizio 4
Grafo orientato con pesi non negativi. Ogni nodo ha un tipo: B = blocco, P = piscina.

Il metodo `piscina_piu_vicina(G)` restituisce per ogni blocco (nodo di tipo B) la lista delle 3 piscine più vicine a lui.

Supponiamo la presenta di un metodo `G.etichetta(i)` che restituisce il tipo del nodo `i`

In [ ]:
def piscina_piu_vicina(G):
    M = floyd(G)
    piscine = []
    blocchi = []
    for i in range(G.n):
        etichetta = G.etichetta(i)
        if etichetta == "P":
            piscine.append(i)
        else:
            blocchi.append(i)
    risultato = []
    for b in blocchi:
        migliori_piscine = [-1, -1, -1]
        migliori_distanze = [float("inf") for _ in range(2)]
        for p in piscine:
            distanza = M[b][p]
            if distanza < migliori_distanze[0]:
                migliori_distanze[2] = migliori_distanze[1]
                migliori_distanze[1] = migliori_distanze[0]
                migliori_distanze[0] = distanza
                migliori_piscine[2] = migliori_piscine[1]
                migliori_piscine[1] = migliori_piscine[0]
                migliori_piscine[0] = p
            elif distanza < migliori_distanze[1]:
                migliori_distanze[2] = migliori_distanze[1]
                migliori_distanze[1] = distanza
                migliori_piscine[2] = migliori_piscine[1]
                migliori_piscine[1] = p
            elif distanza < migliori_distanze[2]:
                migliori_distanze[2] = distanza
                migliori_piscine[2] = p
        risultato.append(b,migliori_piscine)
    return risultato

## Complessità
La complessità è dominata da Floyd
quadratico in tutti i casi